In [1]:
# Author : Raghav Gupta
# ============================================================
# Cell 1: Imports, project-root setup, data preparation check,
# and strategy configuration
# ============================================================

import sys
from pathlib import Path
from itertools import product

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


project_root = Path.cwd().resolve()

for parent in [project_root] + list(project_root.parents):
    if (parent / "src").exists():
        project_root = parent
        break
else:
    raise FileNotFoundError(
        "Could not find project root containing src folder. "
        f"Current working directory: {Path.cwd()}"
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Current working directory:", Path.cwd())
print("Project root:", project_root)
print("src exists:", (project_root / "src").exists())


# ============================================================
# StackSats imports
# ============================================================

try:
    from stacksats.runner.core import StrategyRunner, BacktestConfig
except ImportError:
    from stacksats.runner.core import StrategyRunner
    from stacksats.strategy_types import BacktestConfig

from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.mvrv.core import MVRVStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy
from stacksats.strategies.stable.baselines.uniform import UniformStrategy


# ============================================================
# Project imports
# ============================================================
# Import shared project modules through src, so the notebook can reuse
# existing helper functions from strategy_utils and plotting utilities.

from src import config as src_config, data_utils, plots, strategy_utils

# Keep the original variable name used throughout the notebook.
config = src_config

# Optional train/test paths from shared config.
TRAIN_PATH = src_config.TRAIN_PATH
TEST_PATH = src_config.TEST_PATH

# Backward-compatible aliases used by existing cells below.
check_stacksats_data = data_utils.check_stacksats_data
StrategyColumns = plots.StrategyColumns
plot_strategy_full_period = plots.plot_strategy_full_period
export_one_year = strategy_utils.export_one_year


# ============================================================
# StackSats prepared dataset check
# ============================================================
# Important:
# Use the already-prepared StackSats file if it exists.
# Only call check_stacksats_data() if the prepared file is missing.
#
# This avoids failing just because data/raw/brk_metrics.parquet is missing,
# when ~/.stacksats/data/bitcoin_analytics.parquet already exists.

btc_path = config.STACKSATS_DATA_PATH

if btc_path.exists():
    print(f"Using existing prepared StackSats dataset: {btc_path}")
else:
    print(f"Prepared StackSats dataset not found at: {btc_path}")
    print("Trying to prepare it from raw BRK metrics...")

    if not check_stacksats_data(
        path=config.STACKSATS_DATA_PATH,
        raw_path=config.RAW_PATH,
    ):
        raise FileNotFoundError(
            "Could not prepare or load StackSats data.\n"
            f"Prepared path: {config.STACKSATS_DATA_PATH}\n"
            f"Raw path:      {config.RAW_PATH}\n\n"
            "If the prepared file already exists somewhere else, update "
            "STACKSATS_DATA_PATH in src/config.py."
        )

print(f"Final BTC dataset path: {btc_path}")


# ============================================================
# Strategy configuration
# ============================================================

# Budget used per calendar-year evaluation window.
TOTAL_BUDGET_USD = config.TOTAL_BUDGET_USD
SATS_PER_BTC = config.SATS_PER_BTC

# Train and test periods from shared config.
TRAIN_START = f"{config.TRAIN_START_YEAR}-01-01"
TRAIN_END = f"{config.SPLIT_YEAR - 1}-12-31"

TEST_START = f"{config.SPLIT_YEAR}-01-01"
TEST_END = f"{config.TEST_END_YEAR}-12-31"

# Minimum number of rows required for a valid calendar-year evaluation window.
WINDOW_SIZE = config.MIN_DAYS_PER_YEAR

# Default lookbacks used before grid search selects the best combination.
# These values are overwritten later by the best-performing grid result.
MOMENTUM_LOOKBACK = 60
SMA_LOOKBACK = 200
REGIME_LOOKBACK = 200

# # Grid values to test.
# MOMENTUM_LOOKBACK_GRID = [30, 60, 90, 120, 150, 180, 210, 240, 270]
# SMA_LOOKBACK_GRID = [30, 60, 90, 120, 150, 180, 210, 240, 270, 300]
# REGIME_LOOKBACK_GRID = [30, 60, 90, 120, 150, 180, 210, 240, 270, 300]

# More focused but richer lookback grid
MOMENTUM_LOOKBACK_GRID = [
    21, 30, 45, 60, 75, 90, 120, 150, 180
]

SMA_LOOKBACK_GRID = [
    60, 90, 120, 150, 180, 200, 210, 240, 270, 300, 340
]

REGIME_LOOKBACK_GRID = [
    60, 90, 120, 150, 180, 200, 210, 240, 270, 300, 340
]

# Unique lookback values used to create rolling features.
LOOKBACK_DAYS = sorted({
    MOMENTUM_LOOKBACK,
    SMA_LOOKBACK,
    REGIME_LOOKBACK,
})

# Small floor to avoid zero or negative allocation signals.
SIGNAL_FLOOR = 1e-8

# Minimum number of days required for a regime to be evaluated.
MIN_REGIME_DAYS = 20

# Tolerance used to decide whether a result is better, worse, or tied.
STATUS_TOLERANCE_PCT = 1e-6

# Fallback strategy options used if a regime appears in test but was not seen in training.
FALLBACK_STRATEGY_GRID = [
    "sma",
    "stacksats_mvrv_weight",
    "stacksats_momentum_weight",
]


# ============================================================
# Regime-strategy helper configuration
# ============================================================

def resolve_fallback_strategy(fallback_strategy, sma_lookback=None):
    """Convert a fallback strategy option into the actual weight column name."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    if fallback_strategy == "sma":
        return f"sma_{sma_lookback}d_weight"

    return fallback_strategy


def get_fallback_strategy_cols(sma_lookback=None):
    """Return actual fallback strategy weight columns for the selected SMA lookback."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    return [
        resolve_fallback_strategy(fallback_strategy, sma_lookback)
        for fallback_strategy in FALLBACK_STRATEGY_GRID
    ]


def get_candidate_cols(sma_lookback=None):
    """Return candidate strategy columns for the selected SMA lookback."""
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK

    return [
        "stacksats_mvrv_weight",
        "stacksats_momentum_weight",
        f"sma_{sma_lookback}d_weight",
    ]


# Default fallback strategy before grid search overwrites it.
FALLBACK_STRATEGY = resolve_fallback_strategy("sma", SMA_LOOKBACK)

CANDIDATE_COLS = get_candidate_cols(SMA_LOOKBACK)
FALLBACK_CANDIDATE_COLS = get_fallback_strategy_cols(SMA_LOOKBACK)


# ============================================================
# Final setup printout
# ============================================================

print("Train period:", TRAIN_START, "to", TRAIN_END)
print("Test period: ", TEST_START, "to", TEST_END)
print("Total budget USD:", TOTAL_BUDGET_USD)
print("SATS_PER_BTC:", SATS_PER_BTC)
print("Candidate columns:", CANDIDATE_COLS)
print("Fallback columns:", FALLBACK_CANDIDATE_COLS)


Current working directory: c:\Users\ragha\Documents\capstone\AdaptiveSats\notebooks\proposed_strategies
Project root: C:\Users\ragha\Documents\capstone\AdaptiveSats
src exists: True
Using existing prepared StackSats dataset: C:\Users\ragha\.stacksats\data\bitcoin_analytics.parquet
Final BTC dataset path: C:\Users\ragha\.stacksats\data\bitcoin_analytics.parquet
Train period: 2018-01-01 to 2023-12-31
Test period:  2024-01-01 to 2025-12-31
Total budget USD: 1000.0
SATS_PER_BTC: 100000000
Candidate columns: ['stacksats_mvrv_weight', 'stacksats_momentum_weight', 'sma_200d_weight']
Fallback columns: ['sma_200d_weight', 'stacksats_mvrv_weight', 'stacksats_momentum_weight']


In [2]:
# ============================================================
# Cell 2: Helper functions
# ============================================================
# This cell defines general helper functions used across the notebook.
# These functions are reused for labeling results, formatting chart text,
# trimming complete windows, normalizing weights, and summarizing results.

def get_status_from_pct_diff(pct_diff, tolerance=STATUS_TOLERANCE_PCT):
    """
    Convert percentage improvement into a simple status label.

    Parameters
    ----------
    pct_diff : float
        Percentage difference of strategy performance versus DCA.
    tolerance : float
        Small threshold used to avoid classifying tiny numerical differences
        as meaningful wins or losses.

    Returns
    -------
    str
        'better' if strategy beats DCA, 'worse' if it underperforms,
        and 'tie' if the difference is within tolerance.
    """

    # If improvement is greater than tolerance, strategy is better.
    if pct_diff > tolerance:
        return "better"

    # If improvement is below negative tolerance, strategy is worse.
    if pct_diff < -tolerance:
        return "worse"

    # Otherwise treat it as no meaningful difference.
    return "tie"


def get_calendar_year_windows(
    df: pd.DataFrame,
    split_start: str,
    split_end: str,
    min_days: int = WINDOW_SIZE,
):
    """
    Build one evaluation window per calendar year.

    This is used instead of fixed 365-row chunking so leap years are handled
    correctly. For example, the 2024 test window is 2024-01-01 to 2024-12-31
    with 366 rows, and the next test window starts on 2025-01-01.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with a datetime `date` column.
    split_start : str
        Inclusive split start date, such as TRAIN_START or TEST_START.
    split_end : str
        Inclusive split end date, such as TRAIN_END or TEST_END.
    min_days : int
        Minimum rows required to keep a calendar-year window. The default is
        365, so normal years and leap years are both valid.

    Returns
    -------
    tuple[pd.DataFrame, list[dict], pd.DataFrame, pd.DataFrame]
        Raw split dataframe, list of calendar-year window dictionaries,
        metadata for kept windows, and metadata for skipped years.
    """

    split_start_ts = pd.to_datetime(split_start)
    split_end_ts = pd.to_datetime(split_end)

    raw_split = (
        df[
            (df["date"] >= split_start_ts) &
            (df["date"] <= split_end_ts)
        ]
        .copy()
        .sort_values("date")
        .reset_index(drop=True)
    )

    windows = []
    kept_rows = []
    skipped_rows = []

    for year in range(split_start_ts.year, split_end_ts.year + 1):
        year_start = max(pd.Timestamp(year=year, month=1, day=1), split_start_ts)
        year_end = min(pd.Timestamp(year=year, month=12, day=31), split_end_ts)

        year_df = (
            raw_split[
                (raw_split["date"] >= year_start) &
                (raw_split["date"] <= year_end)
            ]
            .copy()
            .sort_values("date")
            .reset_index(drop=True)
        )

        observed_days = len(year_df)
        expected_days = (year_end - year_start).days + 1

        if observed_days < min_days:
            skipped_rows.append({
                "year": year,
                "start_date": year_start,
                "end_date": year_end,
                "observed_days": observed_days,
                "expected_calendar_days": expected_days,
                "reason": f"less than {min_days} rows",
            })
            continue

        window_number = len(windows) + 1
        windows.append({
            "window": window_number,
            "year": year,
            "start_date": year_df["date"].min(),
            "end_date": year_df["date"].max(),
            "days": observed_days,
            "expected_calendar_days": expected_days,
            "data": year_df,
        })

        kept_rows.append({
            "window": window_number,
            "year": year,
            "start_date": year_df["date"].min(),
            "end_date": year_df["date"].max(),
            "days": observed_days,
            "expected_calendar_days": expected_days,
            "is_leap_window": observed_days == 366,
        })

    window_metadata_df = pd.DataFrame(kept_rows)
    skipped_metadata_df = pd.DataFrame(skipped_rows)

    return raw_split, windows, window_metadata_df, skipped_metadata_df


def concat_calendar_windows(windows):
    """Concatenate the kept calendar-year windows into one dataframe."""

    if not windows:
        return pd.DataFrame()

    return pd.concat(
        [w["data"].copy() for w in windows],
        ignore_index=True,
    )

def build_simple_normalized_weights(signal_multiplier, signal_floor=SIGNAL_FLOOR):
    """
    Convert signal multipliers into normalized daily allocation weights.

    Parameters
    ----------
    signal_multiplier : array-like
        Raw signal strength or multiplier values.
    signal_floor : float
        Minimum allowed signal value to prevent zero or negative weights.

    Returns
    -------
    np.ndarray
        Daily allocation weights that sum to 1.
    """

    # Convert input into a numpy array.
    signal_multiplier = np.asarray(signal_multiplier, dtype=float)

    # Prevent empty signal arrays.
    if len(signal_multiplier) == 0:
        raise ValueError("Empty signal array.")

    # Replace NaN and infinity values with 0.
    clean_signal = np.nan_to_num(
        signal_multiplier,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    # Apply a small floor so no value is exactly zero or negative.
    clean_signal = np.maximum(clean_signal, signal_floor)

    # If all signals are invalid or zero, fall back to uniform weights.
    if clean_signal.sum() <= 0:
        return np.full(len(clean_signal), 1.0 / len(clean_signal))

    # Normalize so all daily weights sum to 1.
    return clean_signal / clean_signal.sum()


def summarize_spd_like_composite(window_summary_df):
    """
    Summarize performance across all 365-day windows.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary dataframe containing strategy_sats, dca_sats,
        strategy_spd, dca_spd, and result columns.

    Returns
    -------
    dict
        Summary metrics including total sats, SPD sums, improvement percentage,
        wins, losses, ties, and win rate.
    """

    # Number of full 365-day windows.
    n_windows = len(window_summary_df)

    # Sum strategy and DCA sats-per-dollar across windows.
    strategy_spd_sum = window_summary_df["strategy_spd"].sum()
    dca_spd_sum = window_summary_df["dca_spd"].sum()

    # Extra sats-per-dollar gained or lost vs DCA.
    extra_spd_sum = strategy_spd_sum - dca_spd_sum

    # Ratio of strategy SPD to DCA SPD.
    spd_ratio = strategy_spd_sum / dca_spd_sum

    # Percentage improvement over DCA.
    improvement_pct = (spd_ratio - 1.0) * 100.0

    # Sum total sats accumulated by strategy and DCA.
    strategy_sats = window_summary_df["strategy_sats"].sum()
    dca_sats = window_summary_df["dca_sats"].sum()

    # Extra sats accumulated vs DCA.
    extra_sats = strategy_sats - dca_sats

    # Count windows where strategy beat, lost to, or tied DCA.
    wins = int((window_summary_df["result"] == "better").sum())
    losses = int((window_summary_df["result"] == "worse").sum())
    ties = int((window_summary_df["result"] == "tie").sum())

    # Window win rate.
    win_rate_pct = wins / n_windows * 100.0 if n_windows > 0 else 0.0

    # Return one summary dictionary.
    return {
        "n_windows": n_windows,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "win_rate_pct": win_rate_pct,

        "strategy_sats": strategy_sats,
        "dca_sats": dca_sats,
        "extra_sats_vs_dca": extra_sats,

        "strategy_spd_sum": strategy_spd_sum,
        "dca_spd_sum": dca_spd_sum,
        "extra_spd_sum_vs_dca": extra_spd_sum,

        "strategy_spd_avg": strategy_spd_sum / n_windows,
        "dca_spd_avg": dca_spd_sum / n_windows,
        "extra_spd_avg_vs_dca": extra_spd_sum / n_windows,

        "spd_ratio": spd_ratio,
        "improvement_pct": improvement_pct,
    }


In [3]:
# ============================================================
# Cell 3: Load BTC data
# ============================================================
# This cell loads the prepared Bitcoin analytics parquet file.
# It checks that all required columns exist before moving forward.

if not btc_path.exists():
    raise FileNotFoundError(f"Could not find: {btc_path}")

btc_df = (
    pl.read_parquet(config.STACKSATS_DATA_PATH)
    .with_columns(pl.col("date").cast(pl.Datetime))
    .sort("date")
)

required_cols = [
    "date",
    "price_usd",
    "mvrv",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

missing_cols = [col for col in required_cols if col not in btc_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns from bitcoin_analytics.parquet: {missing_cols}")

print("Loaded BTC rows:", btc_df.height)

display(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date"),
    )
)


Loaded BTC rows: 5689


min_date,max_date
datetime[μs],datetime[μs]
2010-08-16 00:00:00,2026-03-13 00:00:00


In [4]:
# ============================================================
# Cell 4: StackSats strategy export setup
# ============================================================
# This cell sets up the StackSats strategy runner and strategy objects.
# The export function below uses src.strategy_utils.export_one_year().

runner = StrategyRunner()

stacksats_strategy_objects = {
    "stacksats_mvrv_weight": MVRVStrategy(),
    "stacksats_momentum_weight": MomentumStrategy(),
    "stacksats_uniform_weight": UniformStrategy(),
}

# Cache prevents recomputing weights for the same strategy/window repeatedly.
_export_cache = {}


def export_stacksats_weight_frame_for_window(
    strategy_key: str,
    window_df: pd.DataFrame,
    full_btc_df: pl.DataFrame,
):
    """
    Export StackSats strategy weights for one calendar-year window.

    Refactored:
    - Uses src.strategy_utils.export_one_year()
    - Keeps the same output expected by the rest of this notebook:
      columns: date, raw_weight
    """

    if window_df.empty:
        raise ValueError("window_df is empty.")

    window_df = window_df.copy()
    window_df["date"] = pd.to_datetime(window_df["date"]).dt.normalize()

    window_years = sorted(window_df["date"].dt.year.unique())

    if len(window_years) != 1:
        raise ValueError(
            "export_one_year() expects one calendar-year window. "
            f"Found years: {window_years}"
        )

    year = int(window_years[0])
    window_start = window_df["date"].min()
    window_end = window_df["date"].max()

    cache_key = (
        strategy_key,
        window_start.strftime("%Y-%m-%d"),
        window_end.strftime("%Y-%m-%d"),
        "export_one_year",
    )

    if cache_key in _export_cache:
        return _export_cache[cache_key].copy()

    if strategy_key not in stacksats_strategy_objects:
        raise KeyError(
            f"Unknown strategy_key: {strategy_key}. "
            f"Available keys: {list(stacksats_strategy_objects.keys())}"
        )

    strategy = stacksats_strategy_objects[strategy_key]

    exported_df = export_one_year(
        strategy=strategy,
        btc_data=full_btc_df,
        year=year,
        runner=runner,
    )

    if exported_df is None:
        raise ValueError(
            f"export_one_year returned None for {strategy_key}, year {year}"
        )

    if not isinstance(exported_df, pl.DataFrame):
        exported_df = pl.from_pandas(exported_df)

    if exported_df.is_empty():
        raise ValueError(
            f"export_one_year returned no weights for {strategy_key}, year {year}"
        )

    latest_window_weights = (
        exported_df
        .select(["date", "weight"])
        .sort("date")
        .rename({"weight": "raw_weight"})
        .to_pandas()
    )

    latest_window_weights["date"] = pd.to_datetime(
        latest_window_weights["date"]
    ).dt.normalize()

    latest_window_weights = latest_window_weights[
        (latest_window_weights["date"] >= window_start)
        & (latest_window_weights["date"] <= window_end)
    ].copy()

    latest_window_weights = (
        latest_window_weights
        .sort_values("date")
        .drop_duplicates(subset=["date"], keep="last")
        .reset_index(drop=True)
    )

    if latest_window_weights.empty:
        raise ValueError(
            f"No usable weights for {strategy_key} from "
            f"{window_start.date()} to {window_end.date()}"
        )

    _export_cache[cache_key] = latest_window_weights.copy()

    return latest_window_weights.copy()


In [5]:
# ============================================================
# Cell 5: Feature engineering and simplified regime classification
# ============================================================
# This cell creates reusable functions for:
# 1. engineering rolling lookback features
# 2. assigning the simpler combined regime
# 3. preparing a clean dataframe for any lookback combination

def classify_btc_mvrv_market_cap_regime(
    row,
    momentum_lookback=None,
    sma_lookback=None,
    regime_lookback=None,
):
    """
    Classify each day into a simpler BTC/on-chain regime.

    The regime combines:
    1. BTC trend regime using SMA ratio and returns
    2. MVRV valuation regime
    3. market-cap / realized-cap growth regime

    Returns
    -------
    str
        Combined regime label in the format:
        BTC trend regime | MVRV valuation regime | cap-growth regime
    """

    if momentum_lookback is None:
        momentum_lookback = MOMENTUM_LOOKBACK
    if sma_lookback is None:
        sma_lookback = SMA_LOOKBACK
    if regime_lookback is None:
        regime_lookback = REGIME_LOOKBACK

    # ------------------------------------------------------------
    # BTC trend features
    # ------------------------------------------------------------
    sma_selected_ratio = row[f"price_{sma_lookback}d_sma_ratio"]
    sma_regime_ratio = row[f"price_{regime_lookback}d_sma_ratio"]
    return_momentum = row[f"btc_return_{momentum_lookback}d"]
    return_sma = row[f"btc_return_{sma_lookback}d"]

    # ------------------------------------------------------------
    # On-chain / market features
    # ------------------------------------------------------------
    mvrv = row["mvrv"]
    realized_growth = row["realized_cap_growth_rate"]
    market_growth = row["market_cap_growth_rate"]

    # ------------------------------------------------------------
    # 1. BTC trend regime without drawdown
    # ------------------------------------------------------------
    if sma_regime_ratio >= 1.05 and return_sma > 0:
        btc_regime = "BTC Bull"

    elif sma_selected_ratio <= 0.95 and return_sma < 0:
        btc_regime = "BTC Bear"

    elif sma_selected_ratio < 1.0 and return_momentum > 0:
        btc_regime = "BTC Recovery"

    else:
        btc_regime = "BTC Neutral"

    # ------------------------------------------------------------
    # 2. MVRV valuation regime
    # ------------------------------------------------------------
    if mvrv < 1.0:
        valuation_regime = "Low MVRV"

    elif mvrv > 2.5:
        valuation_regime = "High MVRV"

    else:
        valuation_regime = "Normal MVRV"

    # ------------------------------------------------------------
    # 3. Market-cap / realized-cap growth regime
    # ------------------------------------------------------------
    if realized_growth > market_growth:
        cap_regime = "Realized Growth Leading"

    else:
        cap_regime = "Market Growth Leading"

    # ------------------------------------------------------------
    # Final combined regime
    # ------------------------------------------------------------
    return btc_regime + " | " + valuation_regime + " | " + cap_regime




def prepare_btc_data_for_lookbacks(
    full_btc_df,
    momentum_lookback,
    sma_lookback,
    regime_lookback,
):
    """
    Create rolling features and simplified regimes for one lookback combination.
    """

    lookback_days = sorted({
        momentum_lookback,
        sma_lookback,
        regime_lookback,
    })

    feature_exprs = []

    for d in lookback_days:
        feature_exprs.extend([
            pl.col("price_usd")
            .rolling_mean(window_size=d, min_samples=max(3, int(d * 0.30)))
            .alias(f"price_{d}d_sma"),

            pl.col("price_usd")
            .pct_change(d)
            .alias(f"btc_return_{d}d"),
        ])

    temp_btc_df = full_btc_df.with_columns(feature_exprs)

    ratio_exprs = []

    for d in lookback_days:
        ratio_exprs.append(
            (pl.col("price_usd") / pl.col(f"price_{d}d_sma"))
            .alias(f"price_{d}d_sma_ratio")
        )

    temp_btc_df = temp_btc_df.with_columns(ratio_exprs)

    temp_btc_data = temp_btc_df.to_pandas()
    temp_btc_data["date"] = pd.to_datetime(temp_btc_data["date"])

    feature_cols = [
        "price_usd",
        "mvrv",
        "realized_cap_growth_rate",
        "market_cap_growth_rate",
    ]

    for d in lookback_days:
        feature_cols.extend([
            f"price_{d}d_sma",
            f"price_{d}d_sma_ratio",
            f"btc_return_{d}d",
        ])

    temp_btc_data = (
        temp_btc_data
        .dropna(subset=feature_cols)
        .sort_values("date")
        .reset_index(drop=True)
    )

    temp_btc_data["combined_regime"] = temp_btc_data.apply(
        lambda row: classify_btc_mvrv_market_cap_regime(
            row,
            momentum_lookback=momentum_lookback,
            sma_lookback=sma_lookback,
            regime_lookback=regime_lookback,
        ),
        axis=1,
    )

    return temp_btc_data
